# Stock Market Portfolio Performance & Risk Analysis
This notebook cleans the supplied sample dataset, validates source metrics, computes returns and risk measures, and creates visualizations. The benchmark is explicitly simulated.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('../data/stock_observations.csv')
df = pd.read_csv(DATA, parse_dates=['Date'])
df.head()


In [ ]:
# Data quality checks
print('Shape:', df.shape)
print('Missing values:\n', df.isna().sum())
print('Duplicate ticker/date rows:', df.duplicated(['Ticker','Date']).sum())
print('Non-positive prices:', (df['Close'] <= 0).sum())
print('Negative volume:', (df['Volume'] < 0).sum())


In [ ]:
# Standardize and recalculate core measures from the observed closing prices
df['Ticker'] = df['Ticker'].str.strip().str.upper()
df['Sector'] = df['Sector'].str.strip()
df = df.sort_values(['Ticker','Date']).drop_duplicates(['Ticker','Date']).copy()
df['Period_Return_Calc'] = df.groupby('Ticker')['Close'].pct_change()
df['Running_Peak_Calc'] = df.groupby('Ticker')['Close'].cummax()
df['Drawdown_Calc'] = df['Close'] / df['Running_Peak_Calc'] - 1
df['Year'] = df['Date'].dt.year
df['Quarter'] = 'Q' + df['Date'].dt.quarter.astype(str)
df.to_csv('../data/cleaned_stock_observations.csv', index=False)
df.head()


In [ ]:
# Compare source-provided metrics with recalculated values
validation = df[['Ticker','Date','Reported_Period_Return','Period_Return_Calc',
                 'Reported_Drawdown','Drawdown_Calc']].copy()
validation['Return_Difference'] = validation['Reported_Period_Return'] - validation['Period_Return_Calc']
validation['Drawdown_Difference'] = validation['Reported_Drawdown'] - validation['Drawdown_Calc']
validation.head(15)


In [ ]:
# Stock-level return and risk summary
summary = (df.groupby(['Ticker','Sector'])
    .agg(Start_Price=('Close','first'), End_Price=('Close','last'),
         Max_Drawdown=('Drawdown_Calc','min'), Observations=('Close','count'))
    .reset_index())
summary['Total_Return'] = summary['End_Price'] / summary['Start_Price'] - 1
days = (df.groupby('Ticker')['Date'].max() - df.groupby('Ticker')['Date'].min()).dt.days
summary['CAGR'] = [(r.End_Price/r.Start_Price)**(365.25/days.loc[r.Ticker])-1
                   for r in summary.itertuples()]
market = df[df.Ticker=='MKT'].set_index('Date')['Period_Return_Calc']
betas = {}
for ticker, g in df[df.Ticker!='MKT'].groupby('Ticker'):
    paired = pd.concat([g.set_index('Date')['Period_Return_Calc'], market], axis=1).dropna()
    betas[ticker] = paired.iloc[:,0].cov(paired.iloc[:,1]) / paired.iloc[:,1].var() if len(paired)>1 and paired.iloc[:,1].var()!=0 else np.nan
summary['Beta_vs_MKT'] = summary['Ticker'].map(betas)
summary.sort_values('Total_Return', ascending=False).to_csv('../data/stock_summary_calculated.csv', index=False)
summary.sort_values('Total_Return', ascending=False)


In [ ]:
# Visualization 1: total return by ticker
plot_data = summary.sort_values('Total_Return')
plt.figure(figsize=(10,6))
plt.barh(plot_data['Ticker'], plot_data['Total_Return']*100)
plt.xlabel('Total return (%)'); plt.title('Total Return by Stock (2023–2025)')
plt.tight_layout(); plt.savefig('../visuals/total_return_by_stock.png', dpi=160); plt.show()


In [ ]:
# Visualization 2: closing-price trajectories (each ticker indexed to 100)
plt.figure(figsize=(11,6))
for ticker, g in df.groupby('Ticker'):
    g = g.sort_values('Date')
    plt.plot(g['Date'], g['Close']/g['Close'].iloc[0]*100, marker='o', label=ticker)
plt.ylabel('Indexed price (start = 100)'); plt.title('Relative Price Performance')
plt.legend(ncol=4, fontsize=8); plt.tight_layout()
plt.savefig('../visuals/indexed_price_performance.png', dpi=160); plt.show()


In [ ]:
# Visualization 3: maximum drawdown
risk = summary.sort_values('Max_Drawdown')
plt.figure(figsize=(10,6))
plt.barh(risk['Ticker'], risk['Max_Drawdown']*100)
plt.xlabel('Maximum drawdown (%)'); plt.title('Maximum Drawdown by Stock')
plt.tight_layout(); plt.savefig('../visuals/max_drawdown.png', dpi=160); plt.show()


In [ ]:
# Export summary and sector analysis
sector_summary = summary.groupby('Sector').agg(
    Average_Total_Return=('Total_Return','mean'),
    Average_CAGR=('CAGR','mean'),
    Average_Max_Drawdown=('Max_Drawdown','mean'),
    Ticker_Count=('Ticker','count')).reset_index()
sector_summary.to_csv('../data/sector_summary.csv', index=False)
print('Best total return:', summary.loc[summary.Total_Return.idxmax(), 'Ticker'])
print('Worst total return:', summary.loc[summary.Total_Return.idxmin(), 'Ticker'])
print('Worst maximum drawdown:', summary.loc[summary.Max_Drawdown.idxmin(), 'Ticker'])
sector_summary


## Interpretation cautions
- The PDF has only nine observation dates per symbol, not daily price history. Volatility, beta, and CAGR estimates are therefore based on a very small and irregular sample.
- The MKT series is labelled simulated in the source. It is not an investable or official index.
- Price returns exclude dividends, fees, taxes, and corporate-action adjustments unless already reflected in the supplied prices.
- This is an educational historical analysis, not investment advice.